# Repeated-seed baselines, legacy comparison and matched ablations
Run cells in order. Durable artifacts are written to the configured Drive results folder.


In [ ]:
from pathlib import Path
import os, sys, json, errno, time
DRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

def with_drive_retry(operation, attempts=3):
    for attempt in range(attempts):
        try:
            return operation()
        except OSError as exc:
            disconnected = exc.errno in {errno.ENOTCONN, errno.EIO, errno.ESTALE} or "Transport endpoint" in str(exc)
            if not disconnected or attempt == attempts - 1:
                raise
            print(f"Drive disconnected; remounting before retry {attempt + 1}/{attempts - 1}.", flush=True)
            if drive is not None:
                drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def ensure_drive():
    if drive is not None:
        if not DRIVE.is_dir():
            drive.mount("/content/drive", force_remount=False)
        next(DRIVE.iterdir(), None)
with_drive_retry(ensure_drive)

def resolve_v2():
    configured = os.environ.get("ATLAS_V2_ROOT")
    candidates = ([Path(configured).expanduser()] if configured else []) + [
        DRIVE/"projects/implementation/ATLAS-IDNet/implementation/v2",
        Path.cwd(), *Path.cwd().parents,
    ]
    for candidate in candidates:
        if all((candidate/name).is_file() for name in (
            "src/atlas_v2/data.py", "src/atlas_v2/models.py",
            "src/atlas_v2/experiments.py", "src/atlas_v2/agent.py", "requirements.txt"
        )):
            return candidate.resolve()
    raise FileNotFoundError("Synchronized implementation/v2 source is not visible to this kernel.")

V2 = with_drive_retry(resolve_v2)
sys.path.insert(0, str(V2/"src"))
for module in list(sys.modules):
    if module == "atlas_v2" or module.startswith("atlas_v2."):
        del sys.modules[module]
import atlas_v2
print("Code:", V2)


: 

In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(V2/"requirements.txt")])
import numpy as np, pandas as pd, sklearn, scipy, torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CPU count:", os.cpu_count())


In [ ]:
RESULTS = DRIVE/"projects/implementation/ATLAS-IDNet/results/v2" if DRIVE.exists() else V2/"results"
PUBLICATION_CONFIG_PATH = RESULTS/"publication_config.json"
PUBLICATION_CONFIG = json.loads(PUBLICATION_CONFIG_PATH.read_text())
print(json.dumps(PUBLICATION_CONFIG, indent=2))


In [ ]:
from atlas_v2.models import Detector, matched_width, parameter_count
plan = []
for dataset_name, dataset_protocols in PUBLICATION_CONFIG["protocols"].items():
    natural_meta = json.loads((Path(dataset_protocols["natural"]["prepared"])/"metadata.json").read_text())
    reference = Detector(natural_meta["features"], len(natural_meta["classes"]), "atlas", PUBLICATION_CONFIG["width"])
    target = parameter_count(reference)
    for variant in ["atlas", "legacy", "no_global_context", "no_residual", "mean", "concat", "random_groups", "single_residual", "mlp"]:
        if variant in {"atlas", "legacy"}:
            selected_width = PUBLICATION_CONFIG["width"]
            count = parameter_count(Detector(natural_meta["features"], len(natural_meta["classes"]), variant, selected_width))
        else:
            match = matched_width(natural_meta["features"], len(natural_meta["classes"]), variant, target)
            selected_width, count = match["width"], match["parameters"]
        plan.append({"dataset": dataset_name, "protocol": "natural", "variant": variant,
                     "width": selected_width, "parameters": count, "target_parameters": target,
                     "seeds": len(PUBLICATION_CONFIG["seeds"])})
    for variant in ["atlas", "legacy", "no_global_context", "mean", "extra_trees"]:
        plan.append({"dataset": dataset_name, "protocol": "minority_challenge", "variant": variant,
                     "width": PUBLICATION_CONFIG["width"], "parameters": None, "target_parameters": target,
                     "seeds": len(PUBLICATION_CONFIG["challenge_seeds"])})
plan = pd.DataFrame(plan)
plan.to_csv(RESULTS/"experiment_plan.csv", index=False)
display(plan)


In [ ]:
from atlas_v2.experiments import run
from atlas_v2.models import Detector, parameter_count

def run_resilient(*args, **kwargs):
    try:
        return run(*args, **kwargs)
    except RuntimeError as error:
        if kwargs.get("device") == "auto" and ("CUDA" in str(error) or "cuda" in str(error)):
            print("CUDA run failed; resuming the same portable checkpoint on CPU:", error, flush=True)
            kwargs["device"] = "cpu"
            return run(*args, **kwargs)
        raise

for dataset_name, dataset_protocols in PUBLICATION_CONFIG["protocols"].items():
    natural = dataset_protocols["natural"]
    metadata = json.loads((Path(natural["prepared"])/"metadata.json").read_text())
    target = parameter_count(Detector(metadata["features"], len(metadata["classes"]), "atlas", PUBLICATION_CONFIG["width"]))
    root = RESULTS/dataset_name/"publication"/"natural"
    for variant in ["atlas", "legacy", "no_global_context", "no_residual", "mean", "concat", "random_groups", "single_residual", "mlp"]:
        capacity_target = None if variant in {"atlas", "legacy"} else target
        for seed in PUBLICATION_CONFIG["seeds"]:
            run_resilient(natural["prepared"], root, variant=variant, seed=seed,
                          epochs=PUBLICATION_CONFIG["epochs"], width=PUBLICATION_CONFIG["width"],
                          batch_size=PUBLICATION_CONFIG["batch_size"], patience=PUBLICATION_CONFIG["patience"],
                          loss=PUBLICATION_CONFIG["loss"], device="auto", capacity_target=capacity_target)
    for variant in ["random_forest", "extra_trees", "hist_gradient", "logistic"]:
        for seed in PUBLICATION_CONFIG["seeds"]:
            run_resilient(natural["prepared"], root, variant=variant, seed=seed,
                          epochs=PUBLICATION_CONFIG["epochs"], loss=PUBLICATION_CONFIG["loss"], device="cpu")

    challenge = dataset_protocols["minority_challenge"]
    challenge_root = RESULTS/dataset_name/"publication"/"minority_challenge"
    for variant in ["atlas", "legacy", "no_global_context", "mean", "extra_trees"]:
        for seed in PUBLICATION_CONFIG["challenge_seeds"]:
            run_resilient(challenge["prepared"], challenge_root, variant=variant, seed=seed,
                          epochs=PUBLICATION_CONFIG["epochs"], width=PUBLICATION_CONFIG["width"],
                          batch_size=PUBLICATION_CONFIG["batch_size"], patience=PUBLICATION_CONFIG["patience"],
                          loss=PUBLICATION_CONFIG["loss"], device="auto")


Reusing completed run: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/CICIoT2023/publication/natural/atlas_balanced_42_a66ef4d06f327ab2
Reusing completed run: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/CICIoT2023/publication/natural/atlas_balanced_43_e89c41398d67b36f
Reusing completed run: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/CICIoT2023/publication/natural/atlas_balanced_44_4e5176469f825eb5
Reusing completed run: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/CICIoT2023/publication/natural/atlas_balanced_45_be1c3d94cddc1e2b
Reusing completed run: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/CICIoT2023/publication/natural/atlas_balanced_46_b899f4dc30cea517
Resuming legacy_balanced_42_201b3143a39e6e20 at epoch 7 on cpu
legacy_balanced_42_201b3143a39e6e20 {'epoch': 7, 'train_loss': 0.012772380058910002, 'val_macro_f1': 0.47343395088193735, 'val_weighted_f1': 0

: 

In [ ]:
from atlas_v2.experiments import summarize
for dataset_name in PUBLICATION_CONFIG["protocols"]:
    for protocol_name in ("natural", "minority_challenge"):
        root = RESULTS/dataset_name/"publication"/protocol_name
        print(dataset_name, protocol_name)
        display(summarize(root))
